# Практика. Локальный RAG + RAGAS: Qwen3 и multilingual E5

**Работаем без API-ключей и без удалённых LLM.** Все запросы к языковой модели и эмбеддингам обрабатываются в Google Colab.
**Пайплайн:** TXT → чанки → семантический поиск E5 → ответ локальной Qwen3 → RAGAS → сравнение промптов.

### Три логические роли, одна локальная модель

- **ANSWER** отвечает на вопрос с опорой на найденные фрагменты TXT. Участники меняют только её промпт.
- **GENERATOR** в режиме `generate` через *настоящий `RAGAS TestsetGenerator`* составляет тестовые вопросы и эталонные ответы.
- **JUDGE** через RAGAS оценивает ответы, включая фактическую согласованность с извлечённым контекстом.

**Ограничение:** ради запуска на T4 все три роли выполняет **одна и та же Qwen3-4B-Instruct-2507**, хотя промпты и этапы разные. Такой судья не является независимым и может разделять ошибки отвечающей модели. Для серьёзной оценки нужны отдельная более сильная модель-судья и проверенные людьми эталоны.

### Источники тестов

- `generate` — RAGAS автоматически генерирует вопросы и эталоны из TXT; у небольшой локальной модели это может работать нестабильно.
- `example` — 18 заранее размеченных примеров, включая 4 вопроса, на которые документ не отвечает.
- `upload` — загрузка собственного CSV (`user_input`, `reference`).

**Первый пробный прогон:** выберите `TESTSET_MODE="example"`, `RUN_N=3`, `QUICK_EVAL=True`. Затем попробуйте `generate`, полный тестовый набор и расширенные метрики.


## 0. Основные настройки

1. В Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU** (или GPU с ≥16 ГБ VRAM).
2. Установите зависимости, при необходимости **перезапустите сеанс**, затем запускайте ячейки сверху вниз.
3. Для проверки работы всей схемы начните с `example` и трёх вопросов.
4. Для **автоматической генерации тестов самой RAGAS** переключите `TESTSET_MODE` на `generate`. Сохранённые вопросы не будут меняться от эксперимента к эксперименту.

Внимание: полноценная генерация тестов + 6 LLM-метрик даже на 3 вопросах может быть медленной при локальном запуске. Поэтому предусмотрен быстрый режим оценки.


In [ ]:
# ===== ОСНОВНЫЕ НАСТРОЙКИ =====
TESTSET_MODE = "generate"    # "example" | "generate" | "upload"

# Одна локальная языковая модель используется последовательно во всех ролях.
LOCAL_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MAX_NEW_TOKENS = 850        # увеличить до 1200, если JSON генератора обрывается
MAX_INPUT_TOKENS = 8192     # защита от переполнения контекста на T4

# Автоматические тесты генерирует именно RAGAS TestsetGenerator.
TESTSET_SIZE = 3
REGENERATE_TESTSET = False

RUN_N = 3                   # None = все вопросы
TOP_K = 3
E5_MODEL = "intfloat/multilingual-e5-small"
QUICK_EVAL = True           # True = три ключевые RAGAS-метрики, False = все шесть
INCLUDE_EXTRA_METRICS = False  # Noise Sensitivity (очень медленно)

print("Локальная модель:", LOCAL_MODEL_ID)
print("Embeddings:", E5_MODEL)
print("Источник тестов:", TESTSET_MODE)
print("Количество вопросов:", RUN_N if RUN_N is not None else "все")


## 1. Установка библиотек

Устанавливаем **только локальные** библиотеки: PyTorch/Transformers, bitsandbytes для 4-битной загрузки, SentenceTransformers для E5 и RAGAS. Пакеты OpenAI/GigaChat не используются.

Зафиксированы `ragas==0.4.1` и `langchain-community==0.4.1`, чтобы избежать известной проблемы с отсутствующим модулем VertexAI. После обновления пакетов в Colab может понадобиться перезапуск сеанса.


In [ ]:
%pip install -q "ragas==0.4.1" "langchain-community==0.4.1" \
    "transformers>=4.57,<5" "accelerate>=1.1" "bitsandbytes>=0.45" \
    "sentence-transformers>=3.3,<6" "pandas>=2" matplotlib


In [ ]:
import importlib.metadata as md
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.run_config import RunConfig
print("RAGAS:", md.version("ragas"))
print("LangChain Community:", md.version("langchain-community"))
print("Transformers:", md.version("transformers"))
print("Импорты прошли успешно")


## 2. Загружаем Qwen3 в GPU и создаём локального клиента для RAGAS

Загружаем модель **один раз**, в 4-битном режиме NF4. Для генерации, ответа и оценки используются одни веса, поэтому не нужно держать три LLM в памяти.

Ниже определён `LocalQwenChat` — небольшая обёртка, которая позволяет библиотеке **LangChain / RAGAS** обращаться к нашей локальной Transformers-модели. Она **не отправляет запросы** к внешним LLM API.

> Если Colab сообщает `CUDA out of memory`, перезапустите сеанс, закройте другие GPU-процессы и проверьте, что выбрана T4. На CPU эта модель в 4-битном режиме не запускается.


In [ ]:
import torch
from threading import Lock
from pydantic import Field
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from ragas.llms import LangchainLLMWrapper

if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в Colab: Среда выполнения → Сменить среду выполнения → T4 GPU")
print("GPU:", torch.cuda.get_device_name(0))

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_ID,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
qwen_model.eval()
qwen_lock = Lock()  # RAGAS может запрашивать ответы асинхронно; GPU используем по очереди

class LocalQwenChat(BaseChatModel):
    """Простейший интерфейс LangChain для локальной модели Transformers."""
    max_new_tokens: int = Field(default=850)

    @property
    def _llm_type(self) -> str:
        return "local_qwen_transformers"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        role_mapping = {"system": "system", "human": "user", "ai": "assistant"}
        hf_messages = [
            {"role": role_mapping.get(m.type, "user"), "content": str(m.content)}
            for m in messages
        ]
        prompt = qwen_tokenizer.apply_chat_template(
            hf_messages, tokenize=False, add_generation_prompt=True
        )
        inputs = qwen_tokenizer(prompt, return_tensors="pt")
        if inputs["input_ids"].shape[1] > MAX_INPUT_TOKENS:
            raise ValueError(
                f"Промпт слишком длинный ({inputs['input_ids'].shape[1]} токенов). "
                "Уменьшите длину чанков / TOP_K."
            )
        inputs = {k: v.to(qwen_model.device) for k, v in inputs.items()}
        with qwen_lock, torch.inference_mode():
            tokens = qwen_model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                pad_token_id=qwen_tokenizer.eos_token_id,
            )
        new_tokens = tokens[0, inputs["input_ids"].shape[1]:]
        reply = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=reply))])

# Один клиент, используемый на разных этапах. Эталонные ответы ANSWER не получает.
local_llm = LocalQwenChat(max_new_tokens=MAX_NEW_TOKENS)
answer_model = local_llm
generator_ragas_llm = LangchainLLMWrapper(local_llm)
judge_ragas_llm = LangchainLLMWrapper(local_llm)

print("Qwen3 загружена локально. Проверка короткого запроса:")
print(answer_model.invoke([HumanMessage(content="Ответь одним словом: столица Франции?")]).content)


### Проверка локальности

Все обращения к LLM обрабатывает объект `answer_model`/`judge_ragas_llm`, связанный с загруженной Qwen3. E5 также локальная


## 3. Источник эталонов: загрузка CSV по желанию

- В режиме `example` используется встроенный набор из 18 проверочных вопросов по справочнику.
- В режиме `upload` появится стандартное окно Colab для выбора CSV. Нужны столбцы `user_input` и `reference`.
- В режиме `generate` RAGAS самостоятельно подготовит вопросы и эталоны из TXT.

Важно: в режиме `example` и `upload` автоматическая генерация **не запускается**, но RAGAS по-прежнему оценивает ответы.


In [ ]:
from pathlib import Path

UPLOAD_CSV_PATH = None
if TESTSET_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()   # выбираем файл с проверенными вопросами и эталонами
    if not uploaded:
        raise ValueError("Файл с эталонами не загружен")
    UPLOAD_CSV_PATH = Path(next(iter(uploaded)))
    print("Загружены эталоны:", UPLOAD_CSV_PATH)
elif TESTSET_MODE == "example":
    print("Будет использован встроенный ручной набор из 18 тестов")
else:
    print("Включена автоматическая генерация тестов RAGAS")

## 4. База знаний TXT

Ниже встроен учебный справочник вымышленной компании. Он не содержит реальных корпоративных данных.

При `USE_OWN_TXT=True` можно загрузить собственный TXT. **Если меняете TXT, готовые эталоны уже могут не соответствовать документу** — генерируйте новые тесты (`generate`) или загрузите собственные (`upload`).


In [ ]:
from pathlib import Path

USE_OWN_TXT = False
TXT_PATH = Path("rag_knowledge_base.txt")
DEFAULT_TXT = '# Внутренний справочник компании «Люмен Лаб» (учебный пример)\n\nЭТО ВЫМЫШЛЕННАЯ КОМПАНИЯ И ВЫМЫШЛЕННЫЕ ПРАВИЛА. Все даты, суммы, названия и показатели придуманы специально для демонстрации RAG. Источником истины в этой лабораторной работе является только данный TXT-документ.\n\n## 01. Что делает компания\nКомпания «Люмен Лаб» разрабатывает платформу «Маяк» для мониторинга температуры и влажности на складах продуктов питания. Продукт не управляет холодильными установками, а только собирает показания датчиков, показывает графики и отправляет уведомления о превышении порогов. Пилотный запуск платформы «Маяк» состоялся 12 марта 2025 года.\n\n## 02. Ответственные за продукт\nРуководитель продукта «Маяк» — Алина Котова. За техническую инфраструктуру отвечает инженер Михаил Титов. Заявки на подключение клиентов принимает координатор Елена Левина. Все три роли относятся к продукту «Маяк», а не ко всей компании «Люмен Лаб».\n\n## 03. Тарифы\nТариф «Старт» стоит 1 400 рублей в месяц за один склад и включает до 20 датчиков. Тариф «Команда» стоит 3 900 рублей в месяц за один склад и включает до 100 датчиков. Единовременная стоимость подключения одного склада составляет 6 000 рублей независимо от тарифа. Оплата производится помесячно.\n\n## 04. Сбор данных\nВ стандартном режиме «Маяк» получает показания датчиков каждые 15 минут. В режиме критического наблюдения интервал уменьшается до 2 минут. Изменить режим может администратор склада. Если датчик перестал передавать показания более 30 минут, система создаёт уведомление «Нет связи с датчиком».\n\n## 05. Уведомления\nУведомления о превышении температуры отправляются по электронной почте и в Telegram. SMS-уведомления в текущей версии не поддерживаются. Пользователь может включить уведомления только для конкретного склада, но отдельная настройка на уровне конкретного датчика пока недоступна.\n\n## 06. Поддержка и инциденты\nОбычная техническая поддержка работает по будням с 09:00 до 18:00 по московскому времени. Критические инциденты уровня P1 принимаются круглосуточно. Для P1 первое подтверждение обращения должно поступить в течение 10 минут; целевое время решения — 4 часа. Для P2 первое подтверждение — в течение 1 часа; целевое время решения — до конца следующего рабочего дня.\n\n## 07. Безопасность и журналы\nЖурналы пользовательских действий хранятся 90 дней. Резервное копирование базы данных запускается ежедневно в 02:00 по московскому времени. Доступ к административной панели требует двухфакторной аутентификации. Документ не устанавливает срок хранения резервных копий.\n\n## 08. Интеграции и экспорт\n«Маяк» позволяет выгружать измерения в CSV и получать данные через REST API. Прямой экспорт в XLSX отсутствует. REST API доступен только на тарифе «Команда»; выгрузка CSV доступна на обоих тарифах.\n\n## 09. Доступность и обслуживание\nЦелевой показатель доступности платформы составляет 99,5% в месяц. Плановое техническое обслуживание предусмотрено по четвергам с 21:00 до 22:00 по московскому времени. Уведомление о плановом обслуживании публикуется за 24 часа до начала.\n\n## 10. Подключение нового клиента\nПодключение начинается с проверки списка датчиков, затем проводится тест передачи данных. Стандартное подключение одного склада занимает до 3 рабочих дней после получения корректного списка устройств. Обучающий вебинар для сотрудников клиента длится 90 минут.\n\n## 11. Пилотные клиенты\nК концу июня 2025 года к пилоту «Маяка» подключили 7 складов в трёх регионах. Конкретные названия регионов во внутреннем справочнике не указаны. Число складов не равно числу компаний-клиентов: один клиент может иметь несколько складов.\n\n## 12. Технологии\nВеб-API сервиса разработан на Python с использованием FastAPI. Основная операционная база данных — PostgreSQL. Для фоновых задач используется очередь Redis. Мобильное приложение в учебной версии платформы не предусмотрено.\n\n## 13. Обработка заявок и доступы\nНовый аккаунт администратора склада создаёт координатор после согласования с руководителем продукта. Доступ сотрудников к чужим складам по умолчанию закрыт. При увольнении сотрудника администратор склада должен отключить его учётную запись в течение одного рабочего дня.\n\n## 14. Ограничения справочника\nВ документе НЕТ информации о выручке компании, заработной плате сотрудников, банковских реквизитах, юридическом адресе, телефонном номере технической поддержки, планах выхода на зарубежные рынки и названии генерального директора. Нельзя делать выводы об этих сведениях на основе косвенных данных.\n'

if USE_OWN_TXT:
    from google.colab import files
    uploaded = files.upload()
    TXT_PATH = Path(next(iter(uploaded)))
elif not TXT_PATH.exists():
    TXT_PATH.write_text(DEFAULT_TXT, encoding="utf-8")

knowledge_base = TXT_PATH.read_text(encoding="utf-8")
print("База знаний:", TXT_PATH, "|", len(knowledge_base), "символов")
print(knowledge_base[:650], "...")

## 5. Семантический поиск по документу: `multilingual-e5-small`

E5 работает в памяти Colab и не требует API. Для корректного поиска добавляем `passage:` к фрагментам документа и `query:` к пользовательским запросам.

Те же E5-эмбеддинги будут использоваться внутри RAGAS для генерации тестов и метрик, связанных с семантической близостью.


In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

sections = [x.strip() for x in re.split(r"(?=^## )", knowledge_base, flags=re.M)
            if x.strip().startswith("## ")]
if not sections:
    sections = [knowledge_base.strip()]
chunks = []
for section in sections:
    if len(section) <= 1000:
        chunks.append(section)
    else:
        for start in range(0, len(section), 850):
            piece = section[start:start + 1000].strip()
            if piece:
                chunks.append(piece)
if not chunks:
    raise ValueError("TXT пуст: искать нечего")

e5 = SentenceTransformer(E5_MODEL, device="cpu")  # экономим GPU для Qwen3

class LocalE5Embeddings(Embeddings):
    """Правильные query:/passage: префиксы для E5 внутри RAGAS."""
    def embed_documents(self, texts):
        return e5.encode(["passage: " + str(t) for t in texts],
                         normalize_embeddings=True, show_progress_bar=False).tolist()

    def embed_query(self, text):
        return e5.encode(["query: " + str(text)],
                         normalize_embeddings=True, show_progress_bar=False)[0].tolist()

local_embeddings = LocalE5Embeddings()
ragas_embeddings = LangchainEmbeddingsWrapper(local_embeddings)
doc_vectors = np.asarray(local_embeddings.embed_documents(chunks), dtype=np.float32)

def retrieve(question, top_k=TOP_K):
    question_vector = np.asarray(local_embeddings.embed_query(question), dtype=np.float32)
    similarities = doc_vectors @ question_vector
    best_ids = np.argsort(similarities)[::-1][:top_k]
    return [chunks[int(i)] for i in best_ids]

print("Чанков:", len(chunks), "| размерность E5:", doc_vectors.shape[1])
for found in retrieve("Сколько хранятся журналы действий пользователей?"):
    print(" -", found.splitlines()[0][:110])


# Часть 1. Тестовый набор

В `generate` **сама библиотека RAGAS** составляет вопросы и эталонные ответы на основе TXT. При этом локальная Qwen3 используется как модель-генератор. Этот этап может быть долгим и иногда не выдаёт корректный JSON — это ограничение небольших локальных моделей, а не ошибка ключа.

В `example` можно сразу проверить весь пайплайн на 18 встроенных вопросах; в `upload` — на своём CSV.


In [ ]:
# Только справочные тесты. Эти строки НЕ подаются модели-ответчику при генерации ответа.
# Если изменили исходный TXT, замените и эталоны.
BUILTIN_EXAMPLES = [
  {
    "id": 1,
    "user_input": "Как называется платформа и для чего она нужна?",
    "reference": "Платформа «Маяк» предназначена для мониторинга температуры и влажности на складах продуктов питания. Она не управляет холодильными установками.",
    "answerable": True,
    "expected_section": "01"
  },
  {
    "id": 2,
    "user_input": "Когда состоялся пилотный запуск «Маяка»?",
    "reference": "12 марта 2025 года.",
    "answerable": True,
    "expected_section": "01"
  },
  {
    "id": 3,
    "user_input": "Кто руководит продуктом «Маяк» и кто отвечает за инфраструктуру?",
    "reference": "Руководитель продукта — Алина Котова; за техническую инфраструктуру отвечает Михаил Титов.",
    "answerable": True,
    "expected_section": "02"
  },
  {
    "id": 4,
    "user_input": "Сколько стоит тариф «Старт» и сколько датчиков в нём доступно?",
    "reference": "1 400 рублей в месяц за один склад; до 20 датчиков.",
    "answerable": True,
    "expected_section": "03"
  },
  {
    "id": 5,
    "user_input": "Сколько стоит подключить один склад на тарифе «Команда»?",
    "reference": "Единовременное подключение стоит 6 000 рублей независимо от тарифа.",
    "answerable": True,
    "expected_section": "03"
  },
  {
    "id": 6,
    "user_input": "Как часто поступают показания в стандартном и критическом режимах?",
    "reference": "В стандартном режиме каждые 15 минут, в критическом каждые 2 минуты.",
    "answerable": True,
    "expected_section": "04"
  },
  {
    "id": 7,
    "user_input": "Какими каналами отправляются уведомления о температуре? Поддерживаются ли SMS?",
    "reference": "Уведомления отправляются по электронной почте и в Telegram; SMS не поддерживаются.",
    "answerable": True,
    "expected_section": "05"
  },
  {
    "id": 8,
    "user_input": "За сколько должны подтвердить обращение P1 и какое целевое время его решения?",
    "reference": "Подтверждение P1 — в течение 10 минут; целевое время решения — 4 часа.",
    "answerable": True,
    "expected_section": "06"
  },
  {
    "id": 9,
    "user_input": "Как долго хранятся журналы действий и когда делают резервные копии?",
    "reference": "Журналы хранятся 90 дней, резервные копии базы запускаются ежедневно в 02:00 по московскому времени.",
    "answerable": True,
    "expected_section": "07"
  },
  {
    "id": 10,
    "user_input": "Какие способы экспорта данных доступны на тарифе «Старт»?",
    "reference": "На тарифе «Старт» доступна выгрузка CSV, но REST API доступен только на тарифе «Команда», а прямого XLSX-экспорта нет.",
    "answerable": True,
    "expected_section": "08"
  },
  {
    "id": 11,
    "user_input": "Каков целевой uptime и когда проходит плановое обслуживание?",
    "reference": "Целевой показатель доступности — 99,5% в месяц. Плановое обслуживание — по четвергам с 21:00 до 22:00 по Москве.",
    "answerable": True,
    "expected_section": "09"
  },
  {
    "id": 12,
    "user_input": "Сколько рабочих дней занимает подключение и сколько длится вебинар?",
    "reference": "До 3 рабочих дней после получения корректного списка устройств; вебинар длится 90 минут.",
    "answerable": True,
    "expected_section": "10"
  },
  {
    "id": 13,
    "user_input": "Сколько складов участвовало в пилоте к концу июня 2025 года?",
    "reference": "7 складов в трёх регионах; названия регионов не указаны.",
    "answerable": True,
    "expected_section": "11"
  },
  {
    "id": 14,
    "user_input": "Какие технологии используются для API, базы данных и фоновых задач?",
    "reference": "Python и FastAPI для веб-API, PostgreSQL для основной базы, Redis для очереди фоновых задач.",
    "answerable": True,
    "expected_section": "12"
  },
  {
    "id": 15,
    "user_input": "Какой телефон у технической поддержки?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "14"
  },
  {
    "id": 16,
    "user_input": "Как зовут генерального директора «Люмен Лаб»?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "14"
  },
  {
    "id": 17,
    "user_input": "Какую выручку компания получила в 2025 году?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "14"
  },
  {
    "id": 18,
    "user_input": "В каких именно регионах находились семь пилотных складов?",
    "reference": "НЕТ_ДАННЫХ",
    "answerable": False,
    "expected_section": "11"
  }
]

print("Ручных эталонов:", len(BUILTIN_EXAMPLES))

In [ ]:
import pandas as pd
from pathlib import Path

TESTSET_CSV = Path("ragas_local_generated_testset.csv")

if TESTSET_MODE == "example":
    generated_df = pd.DataFrame(BUILTIN_EXAMPLES)
    print("Встроенный набор: вопросов", len(generated_df))

elif TESTSET_MODE == "upload":
    # utf-8-sig позволяет корректно прочитать CSV с русскими буквами из Excel/Colab.
    generated_df = pd.read_csv(UPLOAD_CSV_PATH, encoding="utf-8-sig", keep_default_na=False)
    if "question" in generated_df and "user_input" not in generated_df:
        generated_df = generated_df.rename(columns={"question": "user_input"})
    print("Загружено вопросов:", len(generated_df))

elif TESTSET_MODE == "generate":
    if TESTSET_CSV.exists() and not REGENERATE_TESTSET:
        generated_df = pd.read_csv(TESTSET_CSV, encoding="utf-8-sig")
        print("Загружен сохранённый набор:", len(generated_df))
    else:
        from langchain_core.documents import Document
        from ragas.testset import TestsetGenerator
        from ragas.testset.persona import Persona
        from ragas.testset.transforms.extractors.llm_based import NERExtractor
        from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
        from ragas.testset.synthesizers.prompts import ThemesPersonasMatchingPrompt, PersonaThemesMapping
        from ragas.run_config import RunConfig

        # RAGAS иногда переводит имя персоны, что вызывает KeyError при поиске по имени.
        # Здесь одна персона: темы привязываем к ней детерминированно.
        class ExactPersonaMatchingPrompt(ThemesPersonasMatchingPrompt):
            async def generate(self, data, llm=None, callbacks=None, **kwargs):
                return PersonaThemesMapping(mapping={
                    persona.name: list(data.themes) for persona in data.personas
                })

        source_docs = [Document(page_content=text,
                                metadata={"source": str(TXT_PATH), "section": i+1})
                       for i, text in enumerate(chunks)]
        persona = Persona(name="Сотрудник компании",
                          role_description="Задаёт вопросы по справочнику, тарифам и процессам")
        generator = TestsetGenerator(llm=generator_ragas_llm,
                                     embedding_model=ragas_embeddings,
                                     persona_list=[persona])
        synthesizer = SingleHopSpecificQuerySynthesizer(llm=generator_ragas_llm)
        # Адаптация на русский язык полезна, но для скорости выключена.
        ADAPT_RUSSIAN = False
        if ADAPT_RUSSIAN:
            import asyncio
            ru_prompts = asyncio.run(synthesizer.adapt_prompts("russian", llm=generator_ragas_llm))
            synthesizer.set_prompts(**ru_prompts)
        synthesizer.theme_persona_matching_prompt = ExactPersonaMatchingPrompt()

        # Логическая роль GENERATOR: эталоны формирует локальная Qwen3 через RAGAS.
        testset = generator.generate_with_langchain_docs(
            source_docs,
            testset_size=TESTSET_SIZE,
            transforms=[
                NERExtractor(llm=generator_ragas_llm)
            ],
            transforms_llm=generator_ragas_llm,
            transforms_embedding_model=ragas_embeddings,
            query_distribution=[(synthesizer, 1.0)],
            run_config=RunConfig(
                timeout=300,
                max_workers=1,
                max_retries=1,
            ),
        )
        generated_df = testset.to_pandas().copy()
        generated_df = generated_df.dropna(subset=["user_input", "reference"]).reset_index(drop=True)
        if generated_df.empty:
            raise RuntimeError("Локальный генератор RAGAS не создал валидные тесты. "
                               "Попробуйте увеличить MAX_NEW_TOKENS или используйте TESTSET_MODE='example'.")
        generated_df.insert(0, "id", range(1, len(generated_df)+1))
        generated_df.to_csv(TESTSET_CSV, index=False, encoding="utf-8-sig")
        print("RAGAS сгенерировал и сохранил", len(generated_df), "тестов")
else:
    raise ValueError("TESTSET_MODE должен быть generate, upload или example")

required = {"user_input", "reference"}
if not required.issubset(generated_df.columns):
    raise ValueError(f"В тестах нужны поля {required}, получено {list(generated_df.columns)}")
if "id" not in generated_df.columns:
    generated_df.insert(0, "id", range(1, len(generated_df)+1))
generated_df["id"] = generated_df["id"].astype(str)
generated_df["user_input"] = generated_df["user_input"].astype(str).str.strip()
generated_df["reference"] = generated_df["reference"].astype(str).str.strip()
if generated_df.empty or (generated_df[["user_input", "reference"]] == "").any().any():
    raise ValueError("В CSV отсутствуют вопросы или эталонные ответы")
if generated_df["id"].duplicated().any():
    raise ValueError("В CSV найдены повторяющиеся id")

print("Итоговый набор:", len(generated_df), "вопросов")
display(generated_df[["id", "user_input", "reference"]].head(12))

## 6. Просмотр автоматически сгенерированных или загруженных эталонов

Обязательно проверьте 3–4 пары «вопрос — эталон»: локальный генератор может придумать неверный ответ или плохо сформулировать вопрос. После генерации CSV сохраняется на диске; его можно исправить и использовать в режиме `upload`. **Эталон никогда не отправляется модели-ответчику.**


In [ ]:
for row in generated_df.head(4).itertuples():
    print("\n" + "="*60)
    print("Вопрос:", row.user_input)
    print("Эталон:", row.reference)

if "answerable" in generated_df:
    print("\nВручную размеченные вопросы без ответа:",
          int(generated_df["reference"].eq("НЕТ_ДАННЫХ").sum()))
print("\nЭталон НЕ передаётся модели-ответчику.")

# Часть 2. Самостоятельная практика: улучшаем RAG-промпт

**Менять нужно только `STUDENT_RAG_PROMPT` в следующей ячейке.** Подумайте, как запретить домысливать сведения, научить модель отвечать «НЕТ_ДАННЫХ» и ссылаться на найденные документы.

В промпте обязательно должны остаться `{question}` и `{context}`. JSON-примеры, если будете их добавлять, требуют экранирования фигурных скобок: `{{` и `}}`.


In [13]:
BASELINE_RAG_PROMPT = """
Ответь на вопрос, используя материалы ниже.

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

ОТВЕТ:
"""

# ✏️ САМОСТОЯТЕЛЬНОЕ ЗАДАНИЕ: измените только STUDENT_RAG_PROMPT.
# Подумайте: как не добавлять факты от себя, как давать ссылки на источник,
# когда честнее отказаться, как не пропускать важные детали?
STUDENT_RAG_PROMPT = """
Ты помощник по внутреннему справочнику компании.

КОНТЕКСТ:
{context}

ВОПРОС:
{question}

Ответ:
"""

## 7. Получаем ответы от локальной модели

`RUN_N=3` — быстрая проверка, `RUN_N=None` — все примеры. Для честного сравнения E5 и `TOP_K` одинаковы у обоих промптов. Здесь выполняются **только локальные вызовы** Transformers.


In [ ]:
import pandas as pd
from langchain_core.messages import HumanMessage

selected_tests = generated_df if RUN_N is None else generated_df.head(RUN_N)
if selected_tests.empty:
    raise ValueError("Выберите хотя бы один вопрос")

# Retrieval готовим ОДИН РАЗ, поэтому оба промпта видят идентичные документы.
retrieval_cache = {
    str(row.id): retrieve(row.user_input)
    for row in selected_tests.itertuples()
}

def run_rag(question, prompt_template, contexts):
    rendered = prompt_template.format(
        context="\n\n".join(contexts),
        question=question,
    )
    return str(answer_model.invoke([HumanMessage(content=rendered)]).content).strip()

def run_experiment(prompt_template, title):
    results = []
    for row in selected_tests.itertuples():
        contexts = retrieval_cache[str(row.id)]
        answer = run_rag(row.user_input, prompt_template, contexts)
        results.append({
            "id": str(row.id),
            "user_input": row.user_input,
            "reference": row.reference,
            "retrieved_contexts": contexts,
            "response": answer,
        })
        print(f"{title}: вопрос #{str(row.id)} готов")
    return pd.DataFrame(results)

baseline_df = run_experiment(BASELINE_RAG_PROMPT, "Базовый")
student_df = run_experiment(STUDENT_RAG_PROMPT, "Промпт участника")

preview = student_df[["id", "user_input", "response", "reference"]]
display(preview)

# Часть 3. Оценка качества в RAGAS — тоже локально

Ту же Qwen3 теперь используем в логической роли **JUDGE**. RAGAS формирует проверочные запросы и интерпретирует ответы судьи. Подключения к внешним LLM нет.

**Важно:** судья не является независимым от ответчика — это демонстрационная конфигурация для доступной GPU. Оценки Faithfulness и остальных метрик могут быть нестабильными или `NaN`, если локальная модель не вернула ожидаемый структурированный вывод. Сверяйте подозрительные случаи вручную.


## 7. Как понимать метрики

| Метрика RAGAS | Что проверяет | Лучше |
|---|---|---|
| **Faithfulness** | Какая доля проверяемых утверждений в ответе подтверждается **найденными RAG фрагментами**. **Главная метрика про галлюцинации относительно контекста**. | ↑ к 1 |
| **Factual Correctness (F1)** | Согласованность фактов ответа с **синтетическим эталоном**, баланс ошибок и пропущенных фактов. | ↑ к 1 |
| **Response Relevancy** | Насколько ответ релевантен вопросу. Не гарантирует достоверность! | ↑ к 1 |
| **Context Precision** | Насколько высоко в выдаче ранжируются релевантные куски текста. | ↑ к 1 |
| **Context Recall** | Содержит ли выдача достаточно фактов, необходимых для эталонного ответа. | ↑ к 1 |
| **Semantic Similarity** | Насколько ответ близок по смыслу к эталону (эмбеддинги). Можно быть семантически похожим и фактически ошибочным. | ↑ к 1 |
| **Noise Sensitivity (опционально)** | Насколько ответ допускает ошибки из-за посторонних контекстных фрагментов. | ↓ к 0 |

**Дополнительный показатель в этом Colab:** `1 - Faithfulness` — *приближённая доля неподтверждённых утверждений*, **не** точная вероятность галлюцинации и не доля «ответов с галлюцинациями».

Низкая Faithfulness может означать, что LLM придумала факт **или** что нужный факт просто не попал в retrieved contexts. Именно поэтому сравниваем её с Context Recall. Не усредняйте метрики вслепую: они отвечают на разные вопросы.

**Точность измерения:** все LLM-метрики здесь оценивает та же локальная модель, которая отвечает. `Faithfulness` оценивает *согласованность ответа с retrieved_contexts*, а не объективную истинность каждого факта. `Semantic Similarity` на локальной E5 — только смысловая близость.


In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    SemanticSimilarity,
    NoiseSensitivity,
)
from ragas.metrics.base import MetricWithLLM, MetricWithEmbeddings
from ragas.run_config import RunConfig


def create_metrics():

    if QUICK_EVAL:
        # Быстрая оценка
        metrics = [
            Faithfulness(),
            SemanticSimilarity(),
        ]
    else:
        # Полная оценка
        metrics = [
            Faithfulness(),
            FactualCorrectness(mode="f1"),
            ResponseRelevancy(strictness=1),
            LLMContextPrecisionWithReference(),
            LLMContextRecall(),
            SemanticSimilarity(),
        ]

    if INCLUDE_EXTRA_METRICS:
        metrics.append(
            NoiseSensitivity(mode="irrelevant")
        )

    # Подключаем локальные модели к метрикам
    for metric in metrics:

        if isinstance(metric, MetricWithLLM):
            metric.llm = judge_ragas_llm

        if isinstance(metric, MetricWithEmbeddings):
            metric.embeddings = ragas_embeddings

    return metrics


METRICS = create_metrics()

print("Метрики:", [m.name for m in METRICS])

## 8. Запускаем оценку ответов

RAGAS оценивает два варианта промпта на **одинаковых вопросах, эталонах и retrieved-контекстах**. На локальной Qwen3 рекомендуем `max_workers=1`: параллельная генерация на одной T4 может привести к нехватке видеопамяти. Если метрика не рассчиталась — показываем `NaN`, а не подменяем её нулём.


In [ ]:
# Быстрый режим для локальной LLM
FAST_METRICS = [
    m for m in METRICS
    if m.name in ["faithfulness", "semantic_similarity"]
]

def evaluate_answers(experiment_df, metrics=FAST_METRICS):

    records = experiment_df[
        ["user_input", "retrieved_contexts", "response", "reference"]
    ].to_dict("records")

    dataset = EvaluationDataset.from_list(records)

    result = evaluate(
    dataset=dataset,
    metrics=create_metrics(),
        llm=judge_ragas_llm,
        embeddings=ragas_embeddings,
        run_config=RunConfig(
            timeout=180,
            max_workers=1,
            max_retries=1,
        ),
        raise_exceptions=False,
        show_progress=True,
    )

    scored = result.to_pandas().reset_index(drop=True)
    scored.insert(0, "id", experiment_df["id"].to_list())

    if "faithfulness" in scored.columns:
        scored["unsupported_fraction_estimate"] = (
            1 - pd.to_numeric(
                scored["faithfulness"], errors="coerce"
            )
        )

    return scored


# Сначала проверяем только 2 вопроса
baseline_scores = evaluate_answers(baseline_df.head(2))
student_scores = evaluate_answers(student_df.head(2))

# Показываем результаты
names = [
    m.name for m in FAST_METRICS
] + ["unsupported_fraction_estimate"]

names = [name for name in names if name in student_scores.columns]

display(student_scores[["id", "user_input"] + names])

## 9. Сводные метрики: сравниваем два промпта

Сравнивайте значения и количество **валидно посчитанных** оценок. Для большинства метрик больше — лучше; у `Noise Sensitivity` и `1 - Faithfulness` — наоборот.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rows = []
for name in names:
    before = pd.to_numeric(baseline_scores[name], errors="coerce")
    after = pd.to_numeric(student_scores[name], errors="coerce")
    rows.append({
        "metric": name,
        "baseline": before.mean(),
        "student": after.mean(),
        "change": after.mean() - before.mean(),
        "baseline_valid_n": int(before.notna().sum()),
        "student_valid_n": int(after.notna().sum()),
        "better": "↓" if name.startswith("noise_sensitivity") or name == "unsupported_fraction_estimate" else "↑",
    })
summary = pd.DataFrame(rows)
display(summary.round(3))

ax = summary.set_index("metric")[["baseline", "student"]].plot(
    kind="bar", figsize=(11, 5), rot=35
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Средний балл RAGAS (обычно 0–1)")
ax.set_title("Базовый vs свой промпт: один и тот же тестовый набор")
plt.tight_layout()
plt.show()

## 9. Отдельно оцениваем галлюцинации и необоснованные ответы

`Faithfulness` — оценка доли утверждений, поддержанных **найденными** фрагментами. `1 - Faithfulness` — удобный *индикатор* неподтверждённых утверждений, но не доказанная доля ложных фактов: это оценка LLM-судьи, и она может быть ошибочной.

На примерах с эталоном `НЕТ_ДАННЫХ` считаем **долю правильных отказов** и **долю выдуманных ответов** без участия судьи. Для автоматических тестов RAGAS таких вопросов может не быть, тогда метрики отказов будут не определены.

In [ ]:
import numpy as np

ABSTENTION = "НЕТ_ДАННЫХ"

def add_abstention_stats(df):
    gold_empty = df["reference"].astype(str).str.strip().eq(ABSTENTION)
    response_empty = df["response"].astype(str).str.strip().str.upper().eq(ABSTENTION)
    n_unanswerable = int(gold_empty.sum())
    n_answerable = int((~gold_empty).sum())
    return {
        "n_unanswerable": n_unanswerable,
        "correct_abstention_rate": (float(response_empty[gold_empty].mean()) if n_unanswerable else np.nan),
        "unsupported_answer_rate": (float((~response_empty[gold_empty]).mean()) if n_unanswerable else np.nan),
        "unnecessary_abstention_rate": (float(response_empty[~gold_empty].mean()) if n_answerable else np.nan),
    }

abstention_table = pd.DataFrame([
    {"prompt": "baseline", **add_abstention_stats(baseline_df)},
    {"prompt": "student", **add_abstention_stats(student_df)},
])
display(abstention_table.round(3))
print("Эти метрики проверяют точное слово НЕТ_ДАННЫХ — договоритесь о формате отказа в промпте.")

## 10. Ищем самые подозрительные ответы

Обратите внимание на примеры с **низкой Faithfulness** — это первый сигнал о неподтверждённых утверждениях. Посмотрите на ответ и найденные фрагменты: причина в плохом поиске или модель добавила информацию, которой не было в контексте?

In [ ]:
if "faithfulness" in student_scores.columns:
    review = student_scores.sort_values("faithfulness", na_position="last")
    for row in review.head(3).itertuples(index=False):
        print("\n" + "=" * 70)
        print("ВОПРОС:", row.user_input)
        print("ОТВЕТ RAG:", row.response)
        print("ЭТАЛОН:", row.reference)
        print("FAITHFULNESS:", row.faithfulness)
        for source in row.retrieved_contexts:
            print("  ИСТОЧНИК:", source[:240].replace("\n", " "), "...")
else:
    print("Метрика Faithfulness отсутствует. Проверьте ошибки оценки выше.")

## 10. Сохраняем результаты и собственный фиксированный набор эталонов

Сохраняем текущий тестовый набор, ответы и оценки. Если сейчас использован `generate`, CSV можно **отредактировать вручную и загрузить через `upload`** для следующего прогона. Изменённый промпт не должен изменять вопросы или эталоны в одном сравнении.

In [ ]:
generated_df.to_csv("ragas_current_references.csv", index=False, encoding="utf-8-sig")
baseline_scores.to_csv("ragas_baseline_scores.csv", index=False, encoding="utf-8-sig")
student_scores.to_csv("ragas_student_scores.csv", index=False, encoding="utf-8-sig")
summary.to_csv("ragas_summary.csv", index=False, encoding="utf-8-sig")
abstention_table.to_csv("ragas_abstention_metrics.csv", index=False, encoding="utf-8-sig")
print("Сохранены 5 CSV-файлов с тестами, оценками и метриками отказов.")

## Обсуждение

1. Как изменились **Faithfulness**, **Factual Correctness** и **Context Recall** после нового промпта?
2. Почему плохо найденный фрагмент нельзя исправить одним только промптом?
3. На каких вопросах модель выдумывает факты, а на каких честно отвечает `НЕТ_ДАННЫХ`?
4. Что меняется в оценке, когда **судья и ответчик — одна и та же локальная модель**? Почему оценки нужно проверять человеком?
5. Какие вопросы автоматически создал **RAGAS TestsetGenerator**, насколько они однозначны и подтверждены TXT?

Для полноценного отчёта: `RUN_N=None`, `QUICK_EVAL=False`. Если на этом режиме мало памяти или слишком долго, начните с трёх вопросов.
